In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("skeleton.xlsx")

df.head()

,kode_video,frame,keypoint_id,x,y,Valid Mask
0,P3_S01_R4,0,0,0.0,0.0,False
1,P3_S01_R4,0,1,0.0,0.0,False
2,P3_S01_R4,0,2,0.0,0.0,False
3,P3_S01_R4,0,3,0.0,0.0,False
4,P3_S01_R4,0,4,0.0,0.0,False


In [2]:
sequence_id = "P3_S01_R4"

seq_df = df[df["kode_video"] == sequence_id].copy()

print(seq_df.shape)
seq_df.head()

(7398, 6)


,kode_video,frame,keypoint_id,x,y,Valid Mask
0,P3_S01_R4,0,0,0.0,0.0,False
1,P3_S01_R4,0,1,0.0,0.0,False
2,P3_S01_R4,0,2,0.0,0.0,False
3,P3_S01_R4,0,3,0.0,0.0,False
4,P3_S01_R4,0,4,0.0,0.0,False


In [5]:
T = seq_df["frame"].nunique()
K = seq_df["keypoint_id"].nunique()

print("Frame :", T)
print("Keypoint :", K)

Frame : 137
Keypoint : 54


In [7]:
frames = sorted(seq_df["frame"].unique())
keypoints = sorted(seq_df["keypoint_id"].unique())

skeleton = np.zeros((len(frames), len(keypoints), 2), dtype=np.float32)

for _, row in seq_df.iterrows():

    t = frames.index(row["frame"])
    k = keypoints.index(row["keypoint_id"])

    skeleton[t, k, 0] = row["x"]
    skeleton[t, k, 1] = row["y"]

print(skeleton.shape)

(137, 54, 2)


In [8]:
def missing_keypoint_reconstruction_numpy(origin_input_data):

    result = origin_input_data.copy()

    kp_xy = result.astype(np.float32)

    T, K, _ = kp_xy.shape

    for k in range(K):

        coords = kp_xy[:, k, :]

        valid_mask = ~(
            (coords[:, 0] == 0) &
            (coords[:, 1] == 0)
        )

        valid_idx = np.where(valid_mask)[0]

        if len(valid_idx) == 0:
            continue

        for t in range(T):

            if valid_mask[t]:
                continue

            prev_arr = valid_idx[valid_idx < t]
            next_arr = valid_idx[valid_idx > t]

            if len(prev_arr) and len(next_arr):

                p = prev_arr[-1]
                n = next_arr[0]

                alpha = (t - p) / (n - p)

                coords[t] = (
                    (1 - alpha) * coords[p]
                    + alpha * coords[n]
                )

            elif len(prev_arr):

                coords[t] = coords[prev_arr[-1]]

            elif len(next_arr):

                coords[t] = coords[next_arr[0]]

        kp_xy[:, k, :] = coords

    return kp_xy

In [9]:
k = 10

coords = skeleton[:, k, :]

trace_df = pd.DataFrame({
    "frame": np.arange(len(coords)),
    "x": coords[:,0],
    "y": coords[:,1]
})

trace_df.head(30)

,frame,x,y
0,0,0.0,0.0
1,1,0.0,0.0
2,2,0.0,0.0
3,3,0.0,0.0
4,4,0.0,0.0
5,5,0.0,0.0
6,6,0.0,0.0
7,7,0.0,0.0
8,8,0.0,0.0
9,9,0.0,0.0


In [10]:
missing_frames = np.where(
    (coords[:,0] == 0) &
    (coords[:,1] == 0)
)[0]

print(missing_frames)

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  64  78  79  80  92  93  94  95  96
  97  98  99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114
 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132
 133 134 135 136]


In [11]:
reconstructed = missing_keypoint_reconstruction_numpy(
    skeleton
)

In [12]:
k = 10

before = skeleton[:,k,:]
after = reconstructed[:,k,:]

comparison = pd.DataFrame({
    "frame": np.arange(len(before)),

    "x_before": before[:,0],
    "y_before": before[:,1],

    "x_after": after[:,0],
    "y_after": after[:,1],
})

comparison.loc[
    comparison["x_before"] == 0
].head(20)

,frame,x_before,y_before,x_after,y_after
0,0,0.0,0.0,0.550244,1.003389
1,1,0.0,0.0,0.550244,1.003389
2,2,0.0,0.0,0.550244,1.003389
3,3,0.0,0.0,0.550244,1.003389
4,4,0.0,0.0,0.550244,1.003389
5,5,0.0,0.0,0.550244,1.003389
6,6,0.0,0.0,0.550244,1.003389
7,7,0.0,0.0,0.550244,1.003389
8,8,0.0,0.0,0.550244,1.003389
9,9,0.0,0.0,0.550244,1.003389


In [13]:
def trace_keypoint(skeleton, keypoint_id):

    coords = skeleton[:, keypoint_id, :]

    T = len(coords)

    valid_mask = ~(
        (coords[:,0] == 0) &
        (coords[:,1] == 0)
    )

    valid_idx = np.where(valid_mask)[0]

    for t in range(T):

        if valid_mask[t]:
            continue

        prev_arr = valid_idx[valid_idx < t]
        next_arr = valid_idx[valid_idx > t]

        if len(prev_arr) and len(next_arr):

            p = prev_arr[-1]
            n = next_arr[0]

            alpha = (t-p)/(n-p)

            print("="*50)
            print(f"Missing frame : {t}")
            print(f"Previous frame: {p}")
            print(f"Next frame    : {n}")
            print(f"Alpha         : {alpha:.4f}")
            print(f"Prev coord    : {coords[p]}")
            print(f"Next coord    : {coords[n]}")

In [14]:
trace_keypoint(skeleton, 10)

Missing frame : 64
Previous frame: 63
Next frame    : 65
Alpha         : 0.5000
Prev coord    : [0.55024415 1.0033894 ]
Next coord    : [0.5242244  0.73079664]
Missing frame : 78
Previous frame: 77
Next frame    : 81
Alpha         : 0.2500
Prev coord    : [0.55798227 0.5910191 ]
Next coord    : [0.5514926  0.87956583]
Missing frame : 79
Previous frame: 77
Next frame    : 81
Alpha         : 0.5000
Prev coord    : [0.55798227 0.5910191 ]
Next coord    : [0.5514926  0.87956583]
Missing frame : 80
Previous frame: 77
Next frame    : 81
Alpha         : 0.7500
Prev coord    : [0.55798227 0.5910191 ]
Next coord    : [0.5514926  0.87956583]


In [15]:
df["valid_mask"] = True
df["x_after"] = df["x"]
df["y_after"] = df["y"]

In [17]:
import numpy as np

for k in sorted(df["keypoint_id"].unique()):

    kp_rows = df[df["keypoint_id"] == k].sort_values("frame")

    coords = kp_rows[["x", "y"]].values.astype(np.float32)

    valid_mask = ~(
        (coords[:, 0] == 0) &
        (coords[:, 1] == 0)
    )

    valid_idx = np.where(valid_mask)[0]

    # simpan valid_mask
    df.loc[kp_rows.index, "valid_mask"] = valid_mask

    if len(valid_idx) == 0:
        continue

    coords_after = coords.copy()

    T = len(coords)

    for t in range(T):

        if valid_mask[t]:
            continue

        prev_arr = valid_idx[valid_idx < t]
        next_arr = valid_idx[valid_idx > t]

        if len(prev_arr) and len(next_arr):

            p = prev_arr[-1]
            n = next_arr[0]

            alpha = (t - p) / (n - p)

            coords_after[t] = (
                (1 - alpha) * coords[p]
                + alpha * coords[n]
            )

        elif len(prev_arr):

            coords_after[t] = coords_after[prev_arr[-1]]

        elif len(next_arr):

            coords_after[t] = coords_after[next_arr[0]]

    # simpan hasil
    df.loc[kp_rows.index, "x_after"] = coords_after[:, 0]
    df.loc[kp_rows.index, "y_after"] = coords_after[:, 1]

In [18]:
df["reconstructed"] = (
    ((df["x"] == 0) & (df["y"] == 0))
)

In [19]:
output_file = "skeleton_tracing.xlsx"

df.to_excel(output_file, index=False)

print(f"Saved: {output_file}")

Saved: skeleton_tracing.xlsx
